# E-Commerce Lakehouse: Bronze Payments

## Ingest and validate source payments

Read the original payment CSV with source values preserved as strings.
Add ingestion metadata, save a managed Bronze Delta table, and
compare source and saved row counts.

## Ingest and refresh Bronze payments

Read the complete payments CSV with source values preserved as
strings. Add ingestion metadata and verify 103,886 source records
before replacing the Bronze table. Confirm the saved row count.

In [0]:
from pyspark.sql.functions import current_timestamp

raw_payments_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/ecommerce/raw_files/olist_order_payments_dataset.csv")
)

source_count = raw_payments_df.count()
print("Source payment rows:", source_count)

if source_count != 103886:
    raise ValueError(
        f"Expected 103,886 payment records; found {source_count}. "
        "Stopped before replacing the table."
    )

bronze_payments_df = raw_payments_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

table_name = "workspace.ecommerce.bronze_payments"

(
    bronze_payments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_payments_df = spark.table(table_name)
saved_count = saved_payments_df.count()

if saved_count != source_count:
    raise ValueError("Saved Bronze payment count does not match the source.")

print("Bronze payments refreshed:", saved_count)

Source payment rows: 103886
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Bronze payments table created.
Saved Bronze payment rows: 103886


In [0]:
print("Source payment rows:", raw_payments_df.count())
raw_payments_df.printSchema()

Source payment rows: 103886
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)

